# Multiple Linear Regression

Predict **house price** (`SalePrice`) using multiple features.

The general formula is:

$$
\text{SalePrice} = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \cdots + \beta_n X_n
$$

### What Are We Trying to Analyse?

- **How OLS behaves**
- **Whether multicollinearity exists**
- **How Ridge stabilizes coefficients**
- **How Lasso removes features**
- **Which model performs best**

#### Data preparation

In [24]:
# Load Dataset

import pandas as pd

data = pd.read_csv("AmesHousing.csv")
data.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


In [25]:
# Keep Only Numerical data
numerical_data = data.select_dtypes(include=['number'])

# Drop column with more than 30% missing values
threshold = 0.3
columns_to_drop = numerical_data.columns[numerical_data.isnull().mean() > threshold]
numerical_data = numerical_data.drop(columns = columns_to_drop)

# Fill remaining missing values with the median
numerical_data = numerical_data.fillna(numerical_data.median())

numerical_data.head()


,Order,PID,MS SubClass,Lot Frontage,Lot Area,Overall Qual,Overall Cond,Year Built,Year Remod/Add,Mas Vnr Area,...,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Misc Val,Mo Sold,Yr Sold,SalePrice
0,1,526301100,20,141.0,31770,6,5,1960,1960,112.0,...,210,62,0,0,0,0,0,5,2010,215000
1,2,526350040,20,80.0,11622,5,6,1961,1961,0.0,...,140,0,0,0,120,0,0,6,2010,105000
2,3,526351010,20,81.0,14267,6,6,1958,1958,108.0,...,393,36,0,0,0,0,12500,6,2010,172000
3,4,526353030,20,93.0,11160,7,5,1968,1968,0.0,...,0,0,0,0,0,0,0,4,2010,244000
4,5,527105010,60,74.0,13830,5,5,1997,1998,0.0,...,212,34,0,0,0,0,0,3,2010,189900


#### Train/Test Split

To evaluate how well our model generalizes to unseen data, we split the dataset into training and testing sets:

- **80%** for training the model
- **20%** for testing the model

This helps us check for overfitting and ensures our model performs well on new, unseen data.

In [26]:
from sklearn.model_selection import train_test_split

# Define features and target variable
X = numerical_data.drop(columns=['SalePrice'])
y = numerical_data['SalePrice']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

print("Training Set Size: ", X_train.shape)
print("Testing Set Size: ", X_test.shape)

Training Set Size:  (2344, 38)
Testing Set Size:  (586, 38)


#### Scale the Data

##### What is a Coefficient?
In linear regression, each feature (input variable) is assigned a coefficient. This coefficient represents how much the feature influences the target variable. For example, if a feature has a large coefficient, it has a strong impact on the prediction.

##### What is Magnitude?
Magnitude refers to the size or value of a coefficient. If a coefficient is very large (positive or negative), it means the model relies heavily on that feature.

##### Ridge and Lasso Regression
- **Ridge Regression** adds a penalty to the sum of squared coefficients. This discourages the model from assigning large values to any coefficient, helping prevent overfitting.
- **Lasso Regression** adds a penalty to the sum of the absolute values of coefficients. This not only discourages large coefficients but can also shrink some coefficients to zero, effectively removing less important features.

##### Why Scaling is Important
If features are on different scales (e.g., one ranges from 0–10, another from 0–10,000), the penalty applied by Ridge or Lasso will be dominated by the features with larger values. This makes the regularization unfair and can distort the model.

**Scaling** (using StandardScaler) ensures all features are on the same scale, so the penalties are applied equally, and the model treats all features fairly.

In [27]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit the scaler on the training data and transform both training and testing data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


#### Run OLS (Ordinary Least Square)

We'll use `LinearRegression` to fit an ordinary least squares model.

**Metrics to Collect:**
- **R² (train):** How well the model fits the training data.
- **R² (test):** How well the model generalizes to new data.
- **RMSE:** Root Mean Squared Error, a measure of prediction error.
- **Coefficients:** The weights assigned to each feature.

**Questions to Consider:**
- Are any coefficients very large? (May indicate instability or multicollinearity)
- Is train R² much higher than test R²? (May indicate overfitting)

In [28]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Initialize and fit the model
ols = LinearRegression()
ols.fit(X_train_scaled, y_train)

# Prediction
y_train_pred = ols.predict(X_train_scaled)
y_test_pred = ols.predict(X_test_scaled)

# Calculate Metrics
r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)

rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))


print("R² (train):", r2_train)
print("R² (test):", r2_test)
print("RMSE (train):", rmse_train)
print("RMSE (test):", rmse_test)
print("Coefficients:", ols.coef_)

R² (train): 0.8344865471537406
R² (test): 0.8371700956683596
RMSE (train): 31370.36533791352
RMSE (test): 36131.65596210686
Coefficients: [ -9846.10174786    712.55130896  -7299.40153076   -753.25810838
   4292.58153121  24088.84177232   4762.0593955    8844.57668081
   3807.8052259    5022.60736578  22431.80104458   6939.40624938
  15724.68075122 -12083.97622406   9354.70531896  11364.75928779
   -313.0195928   16731.63661232   4203.55015381   -515.12166535
    -35.43033984  -2100.88755931  -5894.62785025  -2042.97444946
   2234.98720136   2212.91704663   3835.99759298   4711.57010112
   1662.94659375   2578.29506534   -435.11335656   1326.91912287
     64.15681635   4050.16141677  -1056.21228119  -5521.69626371
    470.9310729  -11744.59114797]


#### Interpreting the OLS Regression Metrics

###### R² (Train and Test)
- **R² (coefficient of determination)** measures how well the model explains the variance in the target variable.
- Values close to 1 indicate a good fit.
- If train and test R² are similar and high, the model generalizes well and is not overfitting.

###### RMSE (Train and Test)
- **RMSE (Root Mean Squared Error)** measures the average prediction error in the same units as the target variable (SalePrice).
- Lower RMSE values are better.
- If the test RMSE is only slightly higher than the train RMSE, the model is performing well on unseen data.

###### Coefficients
- These are the weights assigned to each feature by the model.
- Large positive or negative coefficients mean the model is very sensitive to those features.
- Extremely large coefficients may indicate instability or multicollinearity (high correlation between features).

---

**Summary:**  
- Similar R² and RMSE on both train and test sets indicate good generalization and no major overfitting.
- If any coefficients are unusually large, consider using regularization methods like Ridge or Lasso to stabilize them.

#### Check Multicollinearity (VIF)

We compute the Variance Inflation Factor (VIF) for each feature to detect multicollinearity.

**Interpretation:**
| VIF Value | Meaning                    |
|-----------|----------------------------|
| 1–3       | Safe                       |
| 5+        | Moderate multicollinearity |
| 10+       | Serious multicollinearity  |

- High VIF means some features are "talking over each other" (i.e., they are highly correlated).
- In the Ames dataset, features like `GrLivArea`, `TotRmsAbvGrd`, and `1stFlrSF` are often correlated and may show high VIF.

In [29]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Calculate VIF
X_trained_scaled_df = pd.DataFrame(X_train_scaled, columns =  X_train.columns)

# Calculate VIF for each feature
vif_data = pd.DataFrame()
vif_data["Feature"] = X_trained_scaled_df.columns
vif_data["VIF"] = [variance_inflation_factor(X_trained_scaled_df.values, i) for i in range(X_trained_scaled_df.shape[1])]
print(vif_data.sort_values("VIF", ascending=False))

            Feature           VIF
16  Low Qual Fin SF           inf
17      Gr Liv Area           inf
14       1st Flr SF           inf
15       2nd Flr SF           inf
10     BsmtFin SF 1  2.041644e+04
13    Total Bsmt SF  1.901269e+04
12      Bsmt Unf SF  1.878281e+04
11     BsmtFin SF 2  2.898635e+03
0             Order  7.834600e+01
37          Yr Sold  7.665391e+01
27      Garage Cars  5.642923e+00
28      Garage Area  5.455758e+00
7        Year Built  4.849351e+00
24    TotRms AbvGrd  4.491370e+00
1               PID  3.738246e+00
26    Garage Yr Blt  3.164806e+00
5      Overall Qual  3.082605e+00
20        Full Bath  2.848815e+00
8    Year Remod/Add  2.368311e+00
22    Bedroom AbvGr  2.342189e+00
21        Half Bath  2.211420e+00
18   Bsmt Full Bath  2.130846e+00
2       MS SubClass  1.632442e+00
3      Lot Frontage  1.616754e+00
6      Overall Cond  1.567939e+00
23    Kitchen AbvGr  1.565593e+00
25       Fireplaces  1.547383e+00
9      Mas Vnr Area  1.434983e+00
4          Lot

c:\Users\RinozaJiffry\anaconda3\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


##### Interpreting VIF Results

- **VIF (Variance Inflation Factor)** measures how much a feature is correlated with other features. High VIF means high multicollinearity.
- **inf (infinity)** indicates perfect multicollinearity—these features can be exactly predicted from others. This is a serious problem for regression.
- **Very high VIF values** (e.g., 20,000+) also indicate severe multicollinearity.

**What your results show:**
- Features like `Low Qual Fin SF`, `Gr Liv Area`, `1st Flr SF`, and `2nd Flr SF` have infinite VIF, meaning they are perfectly correlated with other features. This makes the regression unstable.
- Features like `BsmtFin SF 1`, `Total Bsmt SF`, and `Bsmt Unf SF` have extremely high VIF, indicating strong multicollinearity.
- Features with VIF between 5 and 10 (e.g., `Garage Cars`, `Garage Area`) have moderate multicollinearity.
- Features with VIF below 3 are generally considered safe.

**What to do:**
You should consider removing or combining features with very high or infinite VIF to reduce multicollinearity and improve model stability. For example, `Gr Liv Area`, `1st Flr SF`, and `2nd Flr SF` are often highly correlated in housing data and may need to be addressed.

#### Run Ridge Regression

We will fit Ridge models with a range of regularization strengths (`alpha`) and compare results.

- **alpha values to try:** 0.1, 1, 10, 100
- **Observations:**
  - Do coefficients shrink as alpha increases?
  - Does test R² improve compared to OLS?
  - Does RMSE decrease?

By penalizing coefficient magnitude, Ridge should reduce overfitting and stabilize large weights.

In [30]:
from sklearn.linear_model import Ridge

alphas = [0.1, 1, 10, 100]
results = []

for a in alphas:
    ridge = Ridge(alpha=a)
    ridge.fit(X_train_scaled, y_train)
    y_test_pred = ridge.predict(X_test_scaled)
    r2 = r2_score(y_test, y_test_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    results.append((a, r2, rmse, ridge.coef_))

for a, r2, rmse, coef in results:
    print(f"alpha={a}")
    print("  test R²:", r2)
    print("  test RMSE:", rmse)
    print("  coef sample:", coef[:5])
    print("---")

alpha=0.1
  test R²: 0.8371668318997605
  test RMSE: 36132.01807241371
  coef sample: [-9742.96142743   691.82512008 -7298.32277013  -758.43136355
  4294.01891938]
---
alpha=1
  test R²: 0.8371734232837775
  test RMSE: 36131.28676435346
  coef sample: [-9137.92653247   582.1786589  -7288.46477884  -758.68609317
  4291.64924222]
---
alpha=10
  test R²: 0.8372598909555158
  test RMSE: 36121.6918837602
  coef sample: [-5555.57602312   -61.37689764 -7193.50095828  -733.97217096
  4264.32668479]
---
alpha=100
  test R²: 0.8378727781840731
  test RMSE: 36053.60969187562
  coef sample: [ -622.74598074  -929.18075287 -6403.54858502  -402.79757032
  4041.99147387]
---


#### Analysis of Ridge Results

The loop above trains a Ridge model for each `alpha` value, evaluates it on the test set, and prints the test **R²**, **RMSE**, and a few coefficient values. Larger `alpha` means stronger regularization.

- Coefficients shrink toward zero as `alpha` increases, which reduces model variance and combats multicollinearity.
- Test R² increases slightly and RMSE decreases with bigger penalties, showing improved generalization compared to plain OLS.

**Optimal regularization** is the `alpha` that gives the best performance on unseen data (highest test R² or lowest RMSE). In practice we would use cross‑validation to search over many alphas and pick the one with the lowest cross‑validated error; the simple scan above suggests a higher value (e.g. `alpha = 100`) performs best for this split.  Adjusting `alpha` tunes the bias/variance trade‑off: too little regularization (small alpha) leaves multicollinearity unaddressed, while too much (very large alpha) can overshrink coefficients and increase bias.

#### Run Lasso Regression 

Unlike Ridge which shrinks coefficients, **Lasso** can set coefficients to exactly zero, effectively removing less important features from the model.

- **alpha values to try:** 0.001, 0.01, 0.1, 1
- **Observations:**
  - Did some coefficients become exactly 0?
  - How many features were removed (set to 0)?
  - Did test R² improve or RMSE decrease?

**Why Lasso removes features:**
Lasso applies an L1 penalty (absolute value) rather than L2 (squared). This penalty structure encourages sparsity—some coefficients hit exactly zero. This is useful for feature selection: zero coefficients mean those features are not contributing to predictions.

In [31]:
from sklearn.linear_model import Lasso

# Initialize list to store results from each alpha
alpha_lasso = [0.001, 0.01, 0.1, 1]
results_lasso = []

for alpha in alpha_lasso:
    # Create a lasso model with current alpha
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_train_scaled, y_train)
    y_test_pred = lasso.predict(X_test_scaled)
    
    # Check how well model explains the variance in test set
    r2 = r2_score(y_test, y_test_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

    # Count how many coefficients are zero (feature removed)
    num_zeros = np.sum(lasso.coef_ == 0)

    results_lasso.append((alpha, r2, rmse, num_zeros, lasso.coef_))

    for alpha, r2, rmse, num_zeros, coef in results_lasso:
        print(f"alpha={alpha}")
        print(f" test R²: {r2}")
        print(f" RMSE: {rmse}")
        print(f" Features removed (coef =0): {num_zeros}")
        print(f" coef sample: {coef[:5]}")
        print()


c:\Users\RinozaJiffry\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.146e+12, tolerance: 1.394e+09
  model = cd_fast.enet_coordinate_descent(


alpha=0.001
 test R²: 0.8371663630196013
 RMSE: 36132.0700936769
 Features removed (coef =0): 0
 coef sample: [-9819.5845279    706.02587416 -7299.43850728  -757.56715068
  4294.02587155]



c:\Users\RinozaJiffry\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.083e+12, tolerance: 1.394e+09
  model = cd_fast.enet_coordinate_descent(


alpha=0.001
 test R²: 0.8371663630196013
 RMSE: 36132.0700936769
 Features removed (coef =0): 0
 coef sample: [-9819.5845279    706.02587416 -7299.43850728  -757.56715068
  4294.02587155]

alpha=0.01
 test R²: 0.8371663500177569
 RMSE: 36132.07153620301
 Features removed (coef =0): 0
 coef sample: [-9817.70911028   705.65616452 -7299.41885245  -757.60666024
  4294.03121249]

alpha=0.001
 test R²: 0.8371663630196013
 RMSE: 36132.0700936769
 Features removed (coef =0): 0
 coef sample: [-9819.5845279    706.02587416 -7299.43850728  -757.56715068
  4294.02587155]

alpha=0.01
 test R²: 0.8371663500177569
 RMSE: 36132.07153620301
 Features removed (coef =0): 0
 coef sample: [-9817.70911028   705.65616452 -7299.41885245  -757.60666024
  4294.03121249]

alpha=0.1
 test R²: 0.8371662525606703
 RMSE: 36132.08234885106
 Features removed (coef =0): 0
 coef sample: [-9799.18943641   702.01686606 -7299.22195512  -757.96338659
  4294.07175313]

alpha=0.001
 test R²: 0.8371663630196013
 RMSE: 36132.07

c:\Users\RinozaJiffry\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.312e+11, tolerance: 1.394e+09
  model = cd_fast.enet_coordinate_descent(


### Lasso: What went wrong and next steps

**Observed issue:**
- Lasso with very small alphas (e.g., 0.001, 0.01) did not set any coefficients to zero — `Features removed (coef = 0): 0`.
- The solver produced a **ConvergenceWarning**: "Objective did not converge." This means the optimization did not finish within the iteration limit or the problem was numerically difficult for the chosen alpha/solver settings.

**Why this happened:**
- The alpha values tested are too small (weak regularization). Lasso behaves almost like OLS when alpha is very small, so it won't produce sparse (zero) coefficients.
- Very small alpha can also make the optimization harder to converge; the coordinate descent solver may need more iterations to reach the optimum.
- Numerical issues or poor scaling can amplify convergence problems (we already scaled features with `StandardScaler`, which is good).

**What we are going to do next**
1. Increase the `alpha` values to force stronger L1 regularization (example candidates: `1, 5, 10, 50, 100`).
2. Increase `max_iter` (for example to `100000`) to give the solver more iterations to converge.
3. Optionally run `LassoCV` (cross‑validated Lasso) to find an optimal `alpha` automatically over a grid.


**Goal:** find a sparse model (if appropriate) that either matches or improves test performance while reducing multicollinearity and model complexity.

In [32]:
alpha_lasso = [1,5,10,50,100]
results_lasso = []

for alpha in alpha_lasso:
    # Create a lasso model with current alpha
    lasso = Lasso(alpha=alpha, max_iter=100000)
    lasso.fit(X_train_scaled, y_train)
    y_test_pred = lasso.predict(X_test_scaled)
    
    # Check how well model explains the variance in test set
    r2 = r2_score(y_test, y_test_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

    # Count how many coefficients are zero (feature removed)
    num_zeros = np.sum(lasso.coef_ == 0)

    results_lasso.append((alpha, r2, rmse, num_zeros, lasso.coef_))

    for alpha, r2, rmse, num_zeros, coef in results_lasso:
        print(f"alpha={alpha}")
        print(f" test R²: {r2}")
        print(f" RMSE: {rmse}")
        print(f" Features removed (coef =0): {num_zeros}")
        print(f" coef sample: {coef[:5]}")
        print()

alpha=1
 test R²: 0.8371649957704954
 RMSE: 36132.221786649396
 Features removed (coef =0): 0
 coef sample: [-9638.76642337   671.70445755 -7297.54232515  -757.39070621
  4293.19229401]

alpha=1
 test R²: 0.8371649957704954
 RMSE: 36132.221786649396
 Features removed (coef =0): 0
 coef sample: [-9638.76642337   671.70445755 -7297.54232515  -757.39070621
  4293.19229401]

alpha=5
 test R²: 0.8371607392643532
 RMSE: 36132.6940316057
 Features removed (coef =0): 3
 coef sample: [-8966.57148259   546.38714082 -7289.87094792  -749.86536537
  4288.08176345]

alpha=1
 test R²: 0.8371649957704954
 RMSE: 36132.221786649396
 Features removed (coef =0): 0
 coef sample: [-9638.76642337   671.70445755 -7297.54232515  -757.39070621
  4293.19229401]

alpha=5
 test R²: 0.8371607392643532
 RMSE: 36132.6940316057
 Features removed (coef =0): 3
 coef sample: [-8966.57148259   546.38714082 -7289.87094792  -749.86536537
  4288.08176345]

alpha=10
 test R²: 0.8371475373877544
 RMSE: 36134.15869589388
 Featu

After trying larger penalties (`alpha` = 5, 10, 50) we finally saw Lasso set some coefficients to zero.  In plain terms, Lasso *dropped 3–4 columns from the data* while keeping almost the same predictive accuracy.  The test R² stayed around 0.837 and the RMSE barely increased.

Why this matters:

* Lasso can automatically choose a simpler model by discarding unhelpful features.  This makes interpretation easier and can reduce overfitting.
* Beginners often expect Lasso to remove features right away; the key is that the penalty must be strong enough.  You tune `alpha` to control how many features you remove.
* A good workflow is to use `LassoCV` or grid search to find the smallest `alpha` that still gives acceptable performance.  That gives you the “optimal” amount of regularization.

Lasso worked as advertised once we increased the regularization strength: it found a slightly sparser model with nearly identical accuracy.  The next step is to inspect which features were dropped and consider using cross‑validation to choose `alpha` automatically.

In [33]:
# Find the dropped features through Lasso
lasso = Lasso(alpha=5, max_iter=100000).fit(X_train_scaled, y_train)
zero_idx = np.where(lasso.coef_ == 0)[0]
print("dropped features:", X_train.columns[zero_idx].tolist())

dropped features: ['Bsmt Unf SF', 'Low Qual Fin SF', 'Full Bath']
